In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("aniketdvd/game-of-thrones-books")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/aniketdvd/game-of-thrones-books


# Collect the data

In [2]:
import os
text_corpus = ''
for dir, _, filenames in os.walk(path):
    for filename in filenames:
        with open(os.path.join(path, filename), 'r', encoding='cp1252') as f:
            text = f.read()
            text_corpus += text

In [3]:
len(text_corpus)

9778274

## Character lvl Tokenizer

In [4]:
# unique characters in the corpus
chars = sorted(list(set(text_corpus)))
vocab_size = len(chars)

# create mapping from chars to int
stoi = { ch: i for i, ch in enumerate(chars) }
itos = { i: ch for i, ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

### Train/Test split

In [5]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [6]:
data = torch.tensor(encode(text_corpus), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

## Defing parameters

In [7]:
batch_size = 64 # no. of sequences per step 
block_size = 256  # context length
max_iters = 5000 # total training steps
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200 # batches used to estimate loss
n_emb = 384 # embedding size (model width)
n_head = 6 # attention heads
n_layer = 6 # transformer layers
dropout = 0.2

### Data Loading

In [8]:
def get_batch(split):
    data = train_data if split == 'train' else test_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [9]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [10]:
class Head(nn.Module):
    """ one head self attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_emb, head_size, bias=False)
        self.query = nn.Linear(n_emb, head_size, bias=False)
        self.value = nn.Linear(n_emb, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B, T, C = x.shape
        k = self.key(x) # (B, T, hs)
        q = self.query(x) # (B, T, hs)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5 # (B, T, hs) @ (B, hs, T) --> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        v = self.value(x) # (B, T, hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) --> (B, T, hs)
        return out
    
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_emb)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out
    
class FeedForward(nn.Module):
    """ a simple linear layer with ReLU """

    def __init__(self, n_emb):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_emb, 4 * n_emb),
            nn.ReLU(),
            nn.Linear(4 * n_emb, n_emb),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [11]:
class Block(nn.Module):
    """ Transformer block """

    def __init__(self, n_emb, n_head):
        super().__init__()
        head_size = n_emb // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_emb)
        self.ln1 = nn.LayerNorm(n_emb)
        self.ln2 = nn.LayerNorm(n_emb)
    
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [12]:
class GPTLanguageModel(nn.Module):
    
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_emb)
        self.position_embedding_table = nn.Embedding(block_size, n_emb)
        # Block 
        self.blocks = nn.Sequential(*[Block(n_emb, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_emb)
        self.lm_head = nn.Linear(n_emb, vocab_size)

        # initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb # (B, T, C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] # (B, C)
            probs = F.softmax(logits, dim=-1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [13]:
model = GPTLanguageModel()
m = model.to(device)

# no. of parameters in model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    
    # get a batch of data
    xb, yb = get_batch('train')
    
    # evaluate loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

10.813537 M parameters
step 0: train loss 4.6574, val loss 4.6578
step 500: train loss 1.8312, val loss 1.8270
step 1000: train loss 1.4481, val loss 1.4522
step 1500: train loss 1.3307, val loss 1.3401
step 2000: train loss 1.2652, val loss 1.2807
step 2500: train loss 1.2279, val loss 1.2469
step 3000: train loss 1.1994, val loss 1.2196
step 3500: train loss 1.1732, val loss 1.1977
step 4000: train loss 1.1544, val loss 1.1806
step 4500: train loss 1.1320, val loss 1.1610
step 4999: train loss 1.1185, val loss 1.1510


In [14]:
# generate from model
context = torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=200)[0].tolist()))


men the champion parted by his own blood … “Oh, ser.” The dungeon sank soundledged up in House 
and the one fell it again. Dareon’s dragon leapt s lerd climb, a greengard. “Hoves teembled again.” 
“Au


In [18]:
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


me to go frightened my garrison, beyond Renly, Arryn.” 
“Then says he wishes and a stout of black white mrice and a tear have receeing the butcher's twins tourney. Unsullied 
his wife, and you have always held on his house. Well, a lot and this lands. The man all wisty, but when 
the Queer outside had stripped of Lamport, Lord Beric had bedressed their godswoman. He stood as quiet 
as direvan thousands to be known the right of suppied." 
"How move they come," Dany told the cuptor. "Pale made a s


## Saving the Model

In [19]:
import torch
import json

# 1. Save model weights (recommended over saving the whole model)
torch.save(model.state_dict(), 'gpt_got.pth')

# 2. Save the tokenizer mappings
tokenizer = {
    'stoi': stoi,
    'itos': itos,
    'vocab_size': vocab_size
}
with open('tokenizer.json', 'w') as f:
    json.dump(tokenizer, f)

# 3. Save hyperparams so you can reconstruct the model architecture
hparams = {
    'block_size': block_size,
    'n_emb': n_emb,
    'n_head': n_head,
    'n_layer': n_layer,
    'dropout': dropout,
    'vocab_size': vocab_size
}
with open('hparams.json', 'w') as f:
    json.dump(hparams, f)

print("Saved: gpt_got.pth, tokenizer.json, hparams.json")

Saved: gpt_got.pth, tokenizer.json, hparams.json
